# 🧬 AnnData Agent
**One command. Smart summary. Free AI via Groq (Llama3-70b).**

### Get your free Groq key
1. Go to **[console.groq.com](https://console.groq.com)**
2. Sign up → click **API Keys** → **Create API Key**
3. Copy key starting with `gsk_...` and paste in the run cell below

---
```python
# With AI:
summarize(adata, groq_api_key='gsk_...')

# Without AI (tables + plots only):
summarize(adata)
```

In [41]:
import numpy as np
import scipy.sparse as sp
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML, clear_output
import warnings, uuid, json
warnings.filterwarnings('ignore')

pio.renderers.default = "notebook_connected"

# ── Theme ──
CARD    = '#ffffff'
BORDER  = '#e4e7f0'
ACCENT  = '#6366f1'
GREEN   = '#10b981'
AMBER   = '#f59e0b'
BLUE    = '#3b82f6'
ROSE    = '#f43f5e'
TEXT    = '#1e293b'
SUBTEXT = '#64748b'
LIGHT   = '#f8fafc'
PALETTE = (
    px.colors.qualitative.Pastel
    + px.colors.qualitative.Set2
    + px.colors.qualitative.Antique
)
PLOT_BASE = dict(
    paper_bgcolor=CARD,
    plot_bgcolor='#f1f5f9',
    font=dict(family='Inter, system-ui, sans-serif', color=TEXT, size=12),
    margin=dict(l=20, r=140, t=55, b=30),
    hoverlabel=dict(bgcolor='#1e293b', font_size=12,
                    font_color='white', font_family='Inter'),
)

GROQ_MODEL = "llama-3.3-70b-versatile"
GROQ_URL   = "https://api.groq.com/openai/v1/chat/completions"

IMPORTANT_KEYWORDS = [
    'cell_type','celltype','cell type','cluster',
    'annotation','label','lineage','subtype','state','population',
    'phenotype','identity','class','polly_raw_cell_type',
    'polly_curated_cell_type','kw_curated_raw_cell_type',
    'sample','donor','patient','subject','individual','participant',
    'disease','condition','diagnosis','status','group','cohort',
    'severity','stage','grade','treatment','therapy','response',
    'age','sex','gender','race','ethnicity',
    'tissue','organ','region','site','location','compartment',
    'batch','library','protocol','platform','run','replicate'
]

SUGGESTED_QUESTIONS = [
    "Which samples dominate the dataset?",
    "Are there any outlier samples?",
    "What are the cell type proportions per condition?",
    "Which tissue has the most diverse cell types?",
    "Summarize the QC metrics",
    "What analysis steps have already been done?",
    "What is the disease vs healthy cell breakdown?",
    "Which cell types are most abundant?",
    "Are there any batch effects I should be aware of?",
    "What would you recommend as the next analysis step?",
]


def _is_important(col):
    return any(kw in col.lower() for kw in IMPORTANT_KEYWORDS)

def _is_cat(s):
    return s.dtype == 'object' or str(s.dtype) == 'category'

def _clean(val):
    return str(val).strip('[]\'\"() ')

def _h(html):
    display(HTML(html))


# ══════════════════════════════════════════════════════
#   summarize() helpers
# ══════════════════════════════════════════════════════

def _header():
    _h(
        '<div style="background:linear-gradient(135deg,#6366f118,#3b82f610);'
        'border:1.5px solid #6366f130;border-radius:14px;'
        'padding:22px 28px 18px;margin:10px 0 18px;font-family:Inter,sans-serif;">'
        '<div style="font-size:24px;font-weight:800;color:#6366f1;">&#x1F9EC; AnnData Agent</div>'
        '<div style="color:#64748b;font-size:13px;margin-top:5px;">'
        'Smart summary &middot; Key columns &middot; AI interpretation via Groq (free)'
        '</div></div>'
    )

def _section(icon, title, color=ACCENT):
    _h(
        '<div style="display:flex;align-items:center;gap:10px;'
        f'border-left:3px solid {color};padding:6px 14px;'
        'margin:22px 0 10px;font-family:Inter,sans-serif;">'
        f'<span style="font-size:16px;">{icon}</span>'
        f'<span style="color:{color};font-weight:700;font-size:14px;'
        f'letter-spacing:.5px;">{title}</span></div>'
    )

def _stat_grid(items):
    chips = ''.join(
        '<div style="background:#ffffff;border:1.5px solid #e4e7f0;border-radius:10px;'
        f'padding:12px 18px;display:inline-block;margin:5px;font-family:Inter,sans-serif;">'
        f'<div style="color:#64748b;font-size:11px;text-transform:uppercase;letter-spacing:.8px;">{k}</div>'
        f'<div style="color:#1e293b;font-size:16px;font-weight:700;margin-top:3px;">{v}</div></div>'
        for k, v in items
    )
    _h(f'<div style="margin:8px 0;">{chips}</div>')

def _cat_table(col, series, color):
    vc    = series.value_counts()
    total = vc.sum()
    uid   = 't' + uuid.uuid4().hex[:8]
    SHOW  = 8
    rows  = []
    for i, (val, cnt) in enumerate(vc.items()):
        pct   = cnt / total * 100
        bar_w = max(4, int(pct * 1.8))
        bg    = LIGHT if i % 2 == 0 else CARD
        hide  = 'display:none;' if i >= SHOW else ''
        rows.append(
            f'<tr class="{uid}-row" style="background:{bg};{hide}">'
            f'<td style="padding:8px 16px;font-size:12.5px;color:{TEXT};'
            f'width:50%;border-right:1px solid {BORDER};word-break:break-word;">{_clean(val)}</td>'
            f'<td style="padding:8px 16px;width:50%;">'
            f'<div style="display:flex;align-items:center;gap:10px;">'
            f'<div style="background:{color};height:7px;width:{bar_w}px;'
            f'border-radius:4px;opacity:.6;flex-shrink:0;"></div>'
            f'<span style="font-size:12.5px;color:{TEXT};font-weight:600;white-space:nowrap;">{cnt:,}</span>'
            f'<span style="font-size:11px;color:{SUBTEXT};white-space:nowrap;">({pct:.1f}%)</span>'
            f'</div></td></tr>'
        )
    toggle = ''
    if len(vc) > SHOW:
        rem = len(vc) - SHOW
        js = (
            f"var r=document.querySelectorAll('.{uid}-row');"
            f"var b=document.getElementById('{uid}-btn');"
            f"var o=b.getAttribute('data-open')==='1';"
            f"r.forEach(function(x,i){{if(i>={SHOW})x.style.display=o?'none':'';}});"
            f"b.setAttribute('data-open',o?'0':'1');"
            f"b.innerText=o?'\\u25BC View all {len(vc)} categories ({rem} more)':'\\u25B2 Show less';"
        )
        toggle = (
            f'<tr><td colspan="2" style="padding:8px 16px;background:{CARD};border-top:1px solid {BORDER};">'
            f'<button id="{uid}-btn" onclick="{js}" data-open="0" '
            f'style="background:none;border:1px solid {color}50;border-radius:6px;'
            f'padding:5px 14px;color:{color};font-size:11.5px;cursor:pointer;font-family:Inter,sans-serif;">'
            f'&#9660; View all {len(vc)} categories ({rem} more)</button></td></tr>'
        )
    _h(
        f'<div style="margin:6px 0 18px;font-family:Inter,sans-serif;">'
        f'<div style="display:flex;align-items:baseline;gap:8px;margin-bottom:8px;">'
        f'<span style="color:{color};font-weight:700;font-size:13px;">{col}</span>'
        f'<span style="color:{SUBTEXT};font-size:11px;">{len(vc)} categories &middot; {total:,} total cells</span>'
        f'</div>'
        f'<table style="border-collapse:collapse;width:100%;table-layout:fixed;'
        f'border:1px solid {BORDER};border-radius:8px;overflow:hidden;">'
        f'<colgroup><col style="width:50%;"><col style="width:50%;"></colgroup>'
        f'<thead><tr style="background:{ACCENT}0f;">'
        f'<th style="text-align:center;padding:8px 16px;font-size:11px;color:{SUBTEXT};'
        f'letter-spacing:.6px;text-transform:uppercase;font-weight:600;'
        f'border-right:1px solid {BORDER};">Value</th>'
        f'<th style="text-align:center;padding:8px 16px;font-size:11px;color:{SUBTEXT};'
        f'letter-spacing:.6px;text-transform:uppercase;font-weight:600;">Count</th>'
        f'</tr></thead>'
        f'<tbody>{"".join(rows)}{toggle}</tbody></table></div>'
    )

def _num_chip(col, series):
    vals = series.dropna()
    _h(
        f'<div style="background:{CARD};border:1px solid {BORDER};'
        f'border-left:3px solid {BLUE};border-radius:8px;'
        f'padding:9px 16px;margin:4px 0 10px;font-family:Inter,sans-serif;">'
        f'<span style="color:{BLUE};font-weight:600;font-size:12px;">{col}</span><br/>'
        f'<span style="color:{SUBTEXT};font-size:11.5px;">'
        f'min <b style="color:{TEXT};">{vals.min():.3g}</b> &nbsp;&middot;&nbsp;'
        f'max <b style="color:{TEXT};">{vals.max():.3g}</b> &nbsp;&middot;&nbsp;'
        f'mean <b style="color:{TEXT};">{vals.mean():.3g}</b> &nbsp;&middot;&nbsp;'
        f'median <b style="color:{TEXT};">{vals.median():.3g}</b> &nbsp;&middot;&nbsp;'
        f'nulls <b style="color:{TEXT};">{series.isna().sum()}</b>'
        f'</span></div>'
    )

def _build_context(adata, key_cols):
    ctx = {
        "n_cells": adata.n_obs, "n_genes": adata.n_vars,
        "embeddings": list(adata.obsm.keys()),
        "layers": list(adata.layers.keys()), "columns": {}
    }
    for col in key_cols:
        s = adata.obs[col]
        if _is_cat(s):
            vc = s.value_counts()
            ctx["columns"][col] = {
                "type": "categorical", "n_unique": len(vc),
                "top_values": {_clean(k): int(v) for k, v in vc.head(5).items()}
            }
        elif np.issubdtype(s.dtype, np.number):
            vals = s.dropna()
            ctx["columns"][col] = {
                "type": "numeric",
                "mean": round(float(vals.mean()), 3),
                "median": round(float(vals.median()), 3),
                "min": round(float(vals.min()), 3),
                "max": round(float(vals.max()), 3),
            }
    return ctx


# ══════════════════════════════════════════════════════
#   GROQ API
# ══════════════════════════════════════════════════════

def _groq_call(messages, groq_api_key, max_tokens=600):
    try:
        import requests
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', '-q'])
        import requests
    resp = requests.post(
        GROQ_URL,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {groq_api_key}",
            "User-Agent": "python-requests/2.31.0",
        },
        json={"model": GROQ_MODEL, "max_tokens": max_tokens,
              "temperature": 0.3, "messages": messages},
        timeout=30,
    )
    if resp.status_code != 200:
        raise Exception(f"HTTP {resp.status_code}: {resp.text[:300]}")
    return resp.json()["choices"][0]["message"]["content"]


def _groq_summary(ctx, groq_api_key):
    prompt = (
        "You are a computational biologist. Summarize this single-cell dataset in 5-6 sentences.\n"
        "Cover: tissue/disease context, dominant cell types, sample structure, QC notes, and one analysis recommendation.\n"
        "Use real numbers. No bullet points. Be direct.\n\nDataset metadata:\n"
        + json.dumps(ctx, indent=2)
    )
    _h(f'<div style="font-family:Inter,sans-serif;color:{SUBTEXT};font-size:13px;'
       'padding:10px 4px;font-style:italic;">&#x2728; Llama-3.3-70b (Groq) is reading your data...</div>')
    try:
        summary = _groq_call([{"role": "user", "content": prompt}], groq_api_key)
        _h(
            f'<div style="background:linear-gradient(135deg,{ACCENT}08,{BLUE}05);'
            f'border:1.5px solid {ACCENT}25;border-radius:12px;'
            'padding:20px 24px;margin:10px 0;font-family:Inter,sans-serif;">'
            '<div style="display:flex;align-items:center;gap:8px;margin-bottom:14px;">'
            f'<span style="font-size:18px;">&#x2728;</span>'
            f'<span style="color:{ACCENT};font-weight:700;font-size:13px;letter-spacing:.4px;">AI INTERPRETATION</span>'
            f'<span style="color:{SUBTEXT};font-size:11px;margin-left:6px;">Llama-3.3-70b via Groq &middot; 100% free</span>'
            '</div>'
            f'<div style="color:{TEXT};font-size:13.5px;line-height:1.85;">{summary}</div></div>'
        )
    except Exception as e:
        _h(
            f'<div style="color:{ROSE};font-family:monospace;font-size:11px;padding:12px 16px;'
            f'border:1px solid {ROSE}40;border-radius:8px;margin:8px 0;white-space:pre-wrap;">'
            f'&#x274C; {str(e)}\n\nFixes:\n'
            f'  - Check key at console.groq.com > API Keys\n'
            f'  - Key format: gsk_xxxxxxxxxxxxxxxxxxxx\n'
            f'  - pip install requests --upgrade</div>'
        )

def _no_key_banner():
    _h(
        f'<div style="background:{AMBER}0f;border:1.5px dashed {AMBER}60;border-radius:10px;'
        'padding:16px 20px;font-family:Inter,sans-serif;margin:8px 0;">'
        f'<div style="font-size:14px;font-weight:700;color:{AMBER};margin-bottom:10px;">'
        '&#x1F511; Add your free Groq key to enable AI interpretation</div>'
        f'<div style="font-size:12.5px;color:{TEXT};line-height:1.9;">'
        '1. Go to <b>console.groq.com</b> &rarr; Sign up (no credit card)<br/>'
        '2. Click <b>API Keys</b> &rarr; <b>Create API Key</b><br/>'
        f'3. Copy key: <code style="background:{BORDER};padding:2px 6px;border-radius:4px;">gsk_...</code><br/>'
        f'4. Run: <code style="background:{BORDER};padding:3px 10px;border-radius:4px;">'
        'summarize(adata, groq_api_key="gsk_...")</code>'
        '</div></div>'
    )

def _done(n_obs, n_vars, n_plots):
    _h(
        f'<div style="background:{GREEN}12;border:1.5px solid {GREEN}40;border-radius:10px;'
        'padding:12px 20px;margin:20px 0 8px;font-family:Inter,sans-serif;'
        'display:flex;align-items:center;gap:12px;">'
        '<span style="font-size:18px;">&#x2705;</span>'
        f'<span style="color:{GREEN};font-weight:700;font-size:13px;">Done!</span>'
        f'<span style="color:{SUBTEXT};font-size:12px;">'
        f'{n_obs:,} cells &middot; {n_vars:,} genes &middot; {n_plots} plots generated'
        '</span></div>'
    )

def set_renderer(r='notebook_connected'):
    pio.renderers.default = r
    print(f'Renderer set to: {r}')


# ══════════════════════════════════════
#   MAIN summarize()
# ══════════════════════════════════════
def summarize(adata, groq_api_key=None, max_unique=50):
    _header()
    _section('📐', 'OVERVIEW')
    x = adata.X
    mtype = (f'Sparse · {x.format.upper()} · {x.dtype}' if sp.issparse(x)
             else f'Dense · {x.dtype}' if x is not None else 'None')
    _stat_grid([
        ('Cells',      f'{adata.n_obs:,}'),
        ('Genes',      f'{adata.n_vars:,}'),
        ('Matrix',     mtype),
        ('Embeddings', ', '.join(adata.obsm.keys()) or '—'),
        ('Layers',     ', '.join(adata.layers.keys()) or '—'),
        ('uns keys',   ', '.join(str(k) for k in list(adata.uns.keys())[:5]) or '—'),
    ])
    all_obs  = adata.obs.columns.tolist()
    key_cols = [c for c in all_obs if _is_important(c)]
    skipped  = [c for c in all_obs if c not in key_cols]
    _section('🔬', f'KEY OBS COLUMNS  ({len(key_cols)} of {len(all_obs)} shown)', GREEN)
    if skipped:
        _h(f'<div style="color:{SUBTEXT};font-size:11.5px;font-family:Inter;'
           f'margin:0 0 10px;padding-left:4px;">&#x2139;&#xFE0F; Other columns not shown: '
           f'<span style="opacity:.6;">{", ".join(skipped)}</span></div>')
    cat_to_plot, num_to_plot, n_plots = [], [], 0
    for col in key_cols:
        s = adata.obs[col]
        if _is_cat(s):
            _cat_table(col, s, GREEN)
            if len(s.value_counts()) <= max_unique:
                cat_to_plot.append(col)
            else:
                _h(f'<div style="color:{AMBER};font-size:11px;font-family:Inter;'
                   f'padding-left:6px;margin:-6px 0 12px;">'
                   f'&#x26A0;&#xFE0F; Plot skipped — too many unique values (>{max_unique})</div>')
        elif np.issubdtype(s.dtype, np.number):
            _num_chip(col, s)
            num_to_plot.append(col)
    _section('✨', 'AI INTERPRETATION  (Llama-3.3-70b via Groq · 100% free)', ACCENT)
    if groq_api_key:
        ctx = _build_context(adata, key_cols)
        _groq_summary(ctx, groq_api_key)
    else:
        _no_key_banner()
    _done(adata.n_obs, adata.n_vars, n_plots)


# ══════════════════════════════════════════════════════════════════
#   SYSTEM PROMPT
# ══════════════════════════════════════════════════════════════════
def _build_system_prompt(adata):
    obs   = adata.obs
    lines = [
        "You are an expert computational biologist assistant.",
        f"The researcher has loaded a single-cell dataset with {adata.n_obs:,} cells and {adata.n_vars:,} genes.",
        f"Available embeddings: {', '.join(adata.obsm.keys()) or 'none'}.",
        f"Available layers: {', '.join(adata.layers.keys()) or 'none'}.",
        "", "Key metadata columns:",
    ]
    for col in obs.columns:
        if not _is_important(col):
            continue
        s = obs[col]
        if _is_cat(s):
            vc    = s.value_counts()
            total = vc.sum()
            parts = ', '.join(
                f'{_clean(k)} ({v:,} cells, {v/total*100:.1f}%)'
                for k, v in vc.head(10).items()
            )
            if len(vc) > 10:
                parts += f' ... +{len(vc)-10} more'
            lines.append(f"  - {col} [{len(vc)} categories]: {parts}")
        elif hasattr(s, 'dtype') and s.dtype.kind in 'iuf':
            vals = s.dropna()
            lines.append(
                f"  - {col} [numeric]: mean={vals.mean():.2g}, "
                f"median={vals.median():.2g}, min={vals.min():.2g}, max={vals.max():.2g}"
            )
    lines += [
        "", "Answer concisely using the numbers above.",
        "Flag samples >2x the median size as dominant/outlier.",
        "If a question can't be answered from metadata, say so clearly.",
        "Keep answers to 3-6 sentences unless detail is requested.",
    ]
    return '\n'.join(lines)


# ══════════════════════════════════════════════════════════════════
#   CHAT  —  clean ipywidgets UI, no resume complexity
#            • 10 clickable question buttons
#            • free-text input + Enter
#            • ■ Stop ends session cleanly (no kernel interrupt)
#            • re-run cell starts a fresh session
# ══════════════════════════════════════════════════════════════════

def _bubble_html(text, role, turn_num=None):
    badge = (f'<span style="font-size:10px;opacity:.4;margin-left:8px;">#{turn_num}</span>'
             if turn_num else '')
    if role == 'user':
        return (
            f'<div style="display:flex;justify-content:flex-end;margin:6px 0;">'
            f'<div style="background:{ACCENT};color:white;border-radius:14px 14px 4px 14px;'
            f'padding:10px 16px;max-width:75%;font-size:13px;line-height:1.5;'
            f'font-family:Inter,sans-serif;word-break:break-word;">{text}{badge}</div></div>'
        )
    return (
        f'<div style="display:flex;justify-content:flex-start;margin:6px 0;">'
        f'<div style="background:{LIGHT};border:1px solid {BORDER};'
        f'border-radius:14px 14px 14px 4px;padding:10px 16px;max-width:80%;'
        f'font-size:13px;line-height:1.6;color:{TEXT};'
        f'font-family:Inter,sans-serif;word-break:break-word;">{text}</div></div>'
    )


def chat(adata, groq_api_key=None):
    """
    Interactive multi-turn chat about your AnnData object.

    • Click any suggested question button to ask instantly
    • Type your own question and press Enter or click Send ↵
    • ■ Stop  — ends the session cleanly (kernel stays free)
    • Re-run this cell to start a brand new chat

    Usage:
        chat(adata, groq_api_key="gsk_...")

    Free key: console.groq.com → API Keys → Create API Key
    """
    try:
        import ipywidgets as widgets
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets', '-q'])
        import ipywidgets as widgets

    if not groq_api_key:
        _h(
            f'<div style="color:{AMBER};font-family:Inter;font-size:13px;padding:12px 16px;'
            f'border:1.5px dashed {AMBER}50;border-radius:8px;line-height:1.8;">'
            f'&#x1F511; Pass your Groq key: '
            f'<code>chat(adata, groq_api_key="gsk_...")</code><br/>'
            f'Free key: <b>console.groq.com</b> &rarr; API Keys &rarr; Create API Key'
            f'</div>'
        )
        return

    system_prompt = _build_system_prompt(adata)
    history       = []       # fresh every call — no resume
    _active       = [True]

    # ── Header ────────────────────────────────────────────────────
    _h(
        f'<div style="background:linear-gradient(135deg,{ACCENT}15,{BLUE}08);'
        f'border:1.5px solid {ACCENT}30;border-radius:14px;'
        f'padding:16px 22px 12px;margin:10px 0 10px;font-family:Inter,sans-serif;">'
        f'<div style="font-size:18px;font-weight:800;color:{ACCENT};">&#x1F4AC; Chat with your data</div>'
        f'<div style="color:{SUBTEXT};font-size:12px;margin-top:5px;">'
        f'{adata.n_obs:,} cells &middot; {adata.n_vars:,} genes &middot; '
        f'Llama-3.3-70b via Groq &middot; 100% free &middot; '
        f're-run cell to start a new chat'
        f'</div></div>'
    )

    # ── Chat output area ──────────────────────────────────────────
    chat_out = widgets.Output(
        layout=widgets.Layout(
            border=f'1.5px solid {BORDER}', border_radius='12px',
            padding='14px', min_height='160px', max_height='460px',
            overflow_y='auto', width='100%',
        )
    )

    # ── Suggested question buttons ────────────────────────────────
    def _make_q_btn(question):
        short = question if len(question) <= 38 else question[:35] + '…'
        return widgets.Button(
            description=short,
            layout=widgets.Layout(width='auto', height='32px', margin='3px'),
            style={'button_color': f'{ACCENT}18', 'font_size': '12px'},
            tooltip=question,
        )

    q_buttons = [_make_q_btn(q) for q in SUGGESTED_QUESTIONS]
    q_label   = widgets.HTML(
        f'<div style="font-family:Inter;font-size:11px;color:{SUBTEXT};'
        f'text-transform:uppercase;letter-spacing:.6px;margin:8px 0 4px;">'
        f'&#x1F4A1; Quick questions — click to ask:</div>'
    )
    q_row1 = widgets.HBox(q_buttons[:5],  layout=widgets.Layout(flex_wrap='wrap', gap='4px'))
    q_row2 = widgets.HBox(q_buttons[5:],  layout=widgets.Layout(flex_wrap='wrap', gap='4px', margin='0 0 4px'))

    # ── Input row ─────────────────────────────────────────────────
    text_input = widgets.Text(
        placeholder='Type your own question and press Enter…',
        layout=widgets.Layout(flex='1', min_width='0'),
        style={'description_width': '0px'},
    )
    send_btn = widgets.Button(
        description='Send ↵',
        layout=widgets.Layout(width='90px', height='32px'),
        style={'button_color': ACCENT, 'font_weight': '600'},
    )
    stop_btn = widgets.Button(
        description='■ Stop',
        button_style='danger',
        layout=widgets.Layout(width='90px', height='32px'),
        tooltip='End session — re-run cell to start a new chat',
    )
    input_row = widgets.HBox(
        [text_input, send_btn, stop_btn],
        layout=widgets.Layout(gap='6px', margin='6px 0 0', width='100%'),
    )
    status_lbl = widgets.HTML(value='')

    panel = widgets.VBox(
        [chat_out, q_label, q_row1, q_row2, input_row, status_lbl],
        layout=widgets.Layout(width='100%'),
    )
    display(panel)

    # ── Helpers ───────────────────────────────────────────────────
    def _add_bubble(text, role, turn_num=None):
        with chat_out:
            display(HTML(_bubble_html(text, role, turn_num)))

    def _set_status(msg, color=SUBTEXT, italic=True):
        style = 'font-style:italic;' if italic else ''
        status_lbl.value = (
            f'<div style="font-family:Inter;font-size:11.5px;color:{color};'
            f'padding:3px 2px;{style}">{msg}</div>'
        )

    def _lock(locked):
        text_input.disabled = locked
        send_btn.disabled   = locked
        for b in q_buttons:
            b.disabled = locked

    # ── Core send ─────────────────────────────────────────────────
    def _send(user_text):
        if not _active[0] or not user_text:
            return
        text_input.value = ''
        _lock(True)
        _set_status('&#x2728; Thinking…', ACCENT)

        turn_num = len(history) // 2 + 1
        _add_bubble(user_text, 'user', turn_num)
        history.append({"role": "user", "content": user_text})

        try:
            answer = _groq_call(
                [{"role": "system", "content": system_prompt}] + history,
                groq_api_key
            )
            history.append({"role": "assistant", "content": answer})
            _add_bubble(answer, 'assistant')
            _set_status(
                f'Model: {GROQ_MODEL} &middot; turn {turn_num} &middot; free via Groq',
                SUBTEXT
            )
        except Exception as e:
            history.pop()
            with chat_out:
                display(HTML(
                    f'<div style="color:{ROSE};font-family:monospace;font-size:11px;'
                    f'white-space:pre-wrap;padding:10px 14px;'
                    f'border:1px solid {ROSE}40;border-radius:8px;">&#x274C; {str(e)}</div>'
                ))
            _set_status('Error — see message above.', ROSE, italic=False)
        finally:
            if _active[0]:
                _lock(False)

    # ── Stop ──────────────────────────────────────────────────────
    def on_stop(_):
        _active[0] = False
        _lock(True)
        stop_btn.disabled = True
        n = len(history) // 2
        _set_status(
            f'&#x23F8;&#xFE0F; Session ended after {n} turn{"s" if n != 1 else ""}. '
            f'Re-run this cell to start a new chat.',
            SUBTEXT, italic=False
        )

    # ── Wire up ───────────────────────────────────────────────────
    send_btn.on_click(lambda _: _send(text_input.value.strip()))
    text_input.on_submit(lambda _: _send(text_input.value.strip()))
    stop_btn.on_click(on_stop)
    for btn, question in zip(q_buttons, SUGGESTED_QUESTIONS):
        btn.on_click(lambda _, q=question: _send(q))


print(f'✅ AnnData Agent loaded!  (renderer: {pio.renderers.default})')
print()
print('  summarize(adata)                           # tables only')
print('  summarize(adata, groq_api_key="gsk_...")   # + AI summary')
print()
print('  chat(adata, groq_api_key="gsk_...")        # open chat widget')
print('    → click any question button to ask instantly')
print('    → type your own question + Enter to send')
print('    → ■ Stop ends session (kernel stays free)')
print('    → re-run cell to start a brand new chat')
print()
print('  Free Groq key: console.groq.com → API Keys → Create API Key')
print('  Plots blank?   set_renderer("notebook") or set_renderer("browser")')

✅ AnnData Agent loaded!  (renderer: notebook_connected)

  summarize(adata)                           # tables only
  summarize(adata, groq_api_key="gsk_...")   # + AI summary

  chat(adata, groq_api_key="gsk_...")        # open chat widget
    → click any question button to ask instantly
    → type your own question + Enter to send
    → ■ Stop ends session (kernel stays free)
    → re-run cell to start a brand new chat

  Free Groq key: console.groq.com → API Keys → Create API Key
  Plots blank?   set_renderer("notebook") or set_renderer("browser")


## 
- download the data from OA
- read the data

In [42]:
# full summary
import scanpy as sc
adata = sc.read_h5ad('GSE233906_GPL19057_raw_custom_processed.h5ad')
summarize(adata, groq_api_key="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")

Value,Count
GSM7438936,"4,483(20.5%)"
GSM7438938,"4,227(19.3%)"
GSM7438939,"3,732(17.1%)"
GSM7438940,"3,451(15.8%)"
GSM7438937,"3,075(14.1%)"
GSM7438935,"2,911(13.3%)"


Value,Count
Total hair loss,"12,161(55.6%)"
No disease,"9,718(44.4%)"


Value,Count
none,"21,879(100.0%)"


Value,Count
0,"3,463(15.8%)"
1,"3,095(14.1%)"
2,"2,316(10.6%)"
3,"2,226(10.2%)"
4,"1,924(8.8%)"
5,"1,685(7.7%)"
6,"1,624(7.4%)"
7,"1,501(6.9%)"
8,993(4.5%)
9,482(2.2%)


Value,Count
CD8+ T cells,"6,042(27.6%)"
Dcs_2,"3,686(16.8%)"
Tregs,"2,226(10.2%)"
Eosinophils/Basophils,"1,924(8.8%)"
Mac/Mono,"1,741(8.0%)"
Dcs_3,"1,685(7.7%)"
NK/NKT,"1,501(6.9%)"
CD4+ T cells/ILCs/γδ T cells,"1,489(6.8%)"
Macs,482(2.2%)
Proliferating,397(1.8%)


...

Value,Count
Female,"21,879(100.0%)"


Value,Count
8 - 10,"21,879(100.0%)"


Value,Count
Primary Site,"12,161(55.6%)"
Normal,"9,718(44.4%)"


Value,Count
weeks,"21,879(100.0%)"


Value,Count
none,"21,879(100.0%)"


- chat with your data

In [ ]:
# chat 
chat(adata, groq_api_key="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")